# 01 分箱模块 (core.binning)

基于真实信贷数据 `hscredit.xlsx`，覆盖所有分箱算法和常用参数。

**数据说明**:
- 12448 条贷款样本，85个字段
- 目标变量: `FPD15`（首期逾期15天坏样本标记）
- 外部评分: `bj_qy24` / `mobtech80` / `bairong_v1` 等
- 行为特征: 网贷次数/金额/逾期记录等 70+ 个特征

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import hscredit as hsc
from hscredit import init_setting, OptimalBinning
from hscredit import (
    UniformBinning, QuantileBinning, TreeBinning, ChiMergeBinning,
    BestKSBinning, BestIVBinning, MDLPBinning, CartBinning,
    KMeansBinning, MonotonicBinning, SmoothBinning,
    BestLiftBinning, TargetBadRateBinning,
)
init_setting()

# 加载真实信贷数据
df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
df['loan_month'] = df['loan_date'].dt.to_period('M').astype(str)
target = 'FPD15'
features = [c for c in df.columns if c not in ['MOB1','MOB2','loan_date','loan_month','FPD15','SFPD15']]
print(f'样本数: {len(df):,}, 坏率: {df[target].mean():.2%}, 特征数: {len(features)}')

ModuleNotFoundError: No module named 'hscredit'

## 1. 快速开始 — 单特征分箱

In [ ]:
feat = 'bj_qy24'
binner = OptimalBinning(method='quantile', max_n_bins=8)
binner.fit(df[[feat]], df[target])
bt = binner.bin_tables_[feat]
print(f'{feat} - IV: {bt["分档IV值"].sum():.4f}')
bt[['分箱标签','样本总数','坏样本率','分档WOE值','分档IV值']]

## 2. 所有分箱方法对比

In [ ]:
feat = 'bj_qy24'
X1 = df[[feat]].dropna(); y1 = df.loc[X1.index, target]
X1 = X1.reset_index(drop=True); y1 = y1.reset_index(drop=True)
methods = {
    'uniform': UniformBinning(n_bins=8),
    'quantile': QuantileBinning(n_bins=8),
    'tree': TreeBinning(max_n_bins=8),
    'chi': ChiMergeBinning(max_n_bins=8),
    'best_ks': BestKSBinning(max_n_bins=8),
    'best_iv': BestIVBinning(max_n_bins=8),
    'mdlp': MDLPBinning(max_n_bins=8),
    'cart': CartBinning(max_n_bins=8),
    'kmeans': KMeansBinning(n_bins=8),
    'smooth': SmoothBinning(max_n_bins=8),
    'best_lift': BestLiftBinning(max_n_bins=8),
    'target_bad_rate': TargetBadRateBinning(max_n_bins=8),
    'monotonic': MonotonicBinning(max_n_bins=8, monotonic='auto'),
}
rows = []
for name, b in methods.items():
    try:
        b.fit(X1, y1)
        bt = b.bin_tables_.get(feat, pd.DataFrame())
        iv_val = bt['分档IV值'].sum() if '分档IV值' in bt.columns else 0
        rows.append({'方法': name, '分箱数': len(bt), 'IV': round(iv_val,4), '最大坏率': round(bt['坏样本率'].max(),4) if '坏样本率' in bt.columns else 0})
    except Exception as e:
        rows.append({'方法': name, '分箱数': 0, 'IV': 0, '最大坏率': str(e)[:40]})
pd.DataFrame(rows).sort_values('IV', ascending=False)

## 3. 单调性约束分箱（外部评分）

In [ ]:
# 外部评分通常应该单调：分越高坏率越低
external_scores = ['bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3']
for score_feat in external_scores:
    valid = df[[score_feat]].dropna()
    y_v = df.loc[valid.index, target].reset_index(drop=True)
    X_v = valid.reset_index(drop=True)
    try:
        b = OptimalBinning(method='monotonic', monotonic='descending', max_n_bins=8)
        b.fit(X_v, y_v)
        bt = b.bin_tables_[score_feat]
        rates = bt['坏样本率'].round(4).tolist()
        print(f'{score_feat:20s} IV={bt["分档IV值"].sum():.4f}  坏率趋势: {rates}')
    except Exception as e:
        print(f'{score_feat}: {e}')

## 4. 批量分箱 — 行为特征

In [ ]:
behavior_feats = [f for f in features if any(k in f for k in ['count','sum','amt','loan_count','lender'])][:12]
batcher = OptimalBinning(method='mdlp', max_n_bins=6)
batcher.fit(df[behavior_feats], df[target])
iv_rows = []
for feat, bt in batcher.bin_tables_.items():
    if '分档IV值' in bt.columns:
        iv_rows.append({'特征': feat, 'IV': round(bt['分档IV值'].sum(),4), '箱数': len(bt)})
pd.DataFrame(iv_rows).sort_values('IV', ascending=False)

## 5. 自定义分割点

In [ ]:
# 业务规则：征信评分按固定切点分箱
b_custom = OptimalBinning(user_splits={'bj_qy24': [550, 580, 610, 640, 670]})
b_custom.fit(df[['bj_qy24']].dropna(), df.loc[df['bj_qy24'].notna(), target])
b_custom.bin_tables_['bj_qy24'][['分箱标签','样本总数','样本占比','坏样本率','分档IV值']]

## 6. 分箱可视化

In [ ]:
from hscredit import bin_plot
b_vis = OptimalBinning(method='mdlp', max_n_bins=8)
b_vis.fit(df[['bj_qy24']], df[target])
fig = bin_plot(b_vis.bin_tables_['bj_qy24'], desc='bj_qy24 征信评分')
plt.show()

## 7. WOE 转换

In [ ]:
top_feats = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','overdue_loan_m1_count_12m']
b_woe = OptimalBinning(method='mdlp', max_n_bins=6)
b_woe.fit(df[top_feats].fillna(-999), df[target])
X_woe = b_woe.transform(df[top_feats].fillna(-999), metric='woe')
print('WOE转换后:')
print(X_woe.head(3))
print('\n各特征WOE取值范围:')
for c in X_woe.columns:
    print(f'  {c}: [{X_woe[c].min():.3f}, {X_woe[c].max():.3f}]')